In [4]:
import pandas as pd 
import re
from lightning import Trainer, LightningModule
from transformers import AutoTokenizer

torch.set_float32_matmul_precision('high')

essential = pd.read_csv("CRISPRInferredCommonEssentials.csv")
# TODO this goes into a preprocessing snakemake
essential["essential"] = True
nonessential = pd.read_csv("AchillesNonessentialControls.csv")
nonessential["essential"] = False

df_essent = pd.concat([essential.rename(columns={"Essentials": "Gene"}), nonessential])
df_essent["quant"] = df_essent["Gene"].apply(lambda v: int(re.search(r"[0-9]+\)$", v).group()[:-2]))
df_essent["Gene"] = df_essent["Gene"].apply(lambda v: re.search(r"^.+ ", v).group()[:-1])

# DepMap provides a list of 1856 genes that are thought to be essential across cancer cell lines, which can be downloaded from https://depmap.org/portal/download/all/?releasename=DepMap+Public+22Q4&filename=CRISPRInferredCommonEssentials.csv . There is also a list of non-essential controls (https://depmap.org/portal/download/all/?releasename=DepMap+Public+22Q4&filename=AchillesNonessentialControls.csv). 


In [2]:
# import from validation now

from single_cellm.jointemb.model import TranscriptomeTextDualEncoderModel, TranscriptomeTextDualEncoderConfig
from single_cellm.jointemb.single_cellm_lightning import TranscriptomeTextDualEncoderLightning
from single_cellm.jointemb.dataset.cancer_gene_essentiality import CancerGeneEssentialityDataModule

pl_model = TranscriptomeTextDualEncoderLightning()
model = pl_model.model
"""
Assess zero-shot cancer gene essentiality prediction performance.

:param model: a trained model

# implementing the easiest
"""

cancer_sentence = "cancer"

# load 10 random transcriptomes from our 15k dataset
datamodule = CancerGeneEssentialityDataModule(
    df_essentiality=df_essent,
    tokenizer="microsoft/biogpt", # model.config.text_config.model_type,
    transcriptome_processor=model.config.transcriptome_config.model_type,
    dataset_name="human_disease",
    batch_size=8,
)


In [13]:
import torch
class LitModule(LightningModule):
    def __init__(self, model, anchor_sentence: str = "cancer"):
        super().__init__()

        self.model = model
        
        tokenizer = AutoTokenizer.from_pretrained("microsoft/biogpt")  # TODO is it biogpt?
        token_dict = tokenizer(anchor_sentence, return_tensors="pt", padding=True)
        self.anchor_embed = self.model.get_text_features(**token_dict, normalize_embeds=True)[1]
        
    def predict_step(self, batch, batch_idx):
        """
        Later use forward() and test_step to directly compute the metric
        """
        labels = batch.pop("labels")
        transcriptome_embeds = self.model.get_transcriptome_features(**batch, normalize_embeds=True)
        
        distances = (
            torch.matmul(self.anchor_embed, transcriptome_embeds.t())
        )
        
        return {
            "labels": labels,
            "predictions": predictions,
            "distances": distances
        }
    

In [14]:
trainer = Trainer()
predictions = trainer.predict(LitModule(model), datamodule=datamodule)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/moritz/.conda/envs/single-cellm6/lib/python3.9/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/moritz/.conda/envs/single-cellm6/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/moritz/.conda/envs/single-cellm6/lib/python3.9/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/moritz/Projects/single-cellm/src/single_cellm/jointemb/geneformer_model.py:97: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be tr

prepared_features has no column attribute 'filter_pass'; tokenizing all cells.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/moritz/.conda/envs/single-cellm6/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Predicting DataLoader 0:   0%|                                                                                                                   | 0/309 [00:00<?, ?it/s]

TypeError: expected Tensor as element 0 in argument 0, but got NoneType

In [ ]:
pl_model = TranscriptomeTextDualEncoderLightning()

In [ ]:
next(loader)


In [ ]:
next(iter(loader))

In [ ]:
len(loader)